# Neo4j Query Graph Pipeline

In [9]:
# 1) Config and imports
import os
import re
import json
from pathlib import Path
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'neo4j://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '12345678')
NEO4J_DB = os.getenv('NEO4J_DB', 'neo4j')

BASE_DIR = Path.cwd()
ANNOT_DIR = BASE_DIR / 'annotation'


In [10]:
# 2) Neo4j helpers and candidate lists
def run_cypher(query, params=None):
    with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD)) as _driver:
        with _driver.session(database=NEO4J_DB) as _session:
            return [r.data() for r in _session.run(query, params or {})]

def fetch_candidates():
    cities = [r['key'] for r in run_cypher('MATCH (c:City) RETURN c.key AS key') if r.get('key')]
    airports = [r['key'] for r in run_cypher('MATCH (a:Airport) RETURN a.key AS key') if r.get('key')]
    countries = [r['key'] for r in run_cypher('MATCH (c:Country) RETURN c.key AS key') if r.get('key')]
    doc_types = [r['v'] for r in run_cypher('MATCH (di:DocumentInstance) WHERE di.documentType IS NOT NULL RETURN DISTINCT di.documentType AS v') if r.get('v')]
    topic_keys = [r['v'] for r in run_cypher('MATCH (q:Questioning) WHERE q.topicKey IS NOT NULL RETURN DISTINCT q.topicKey AS v') if r.get('v')]
    return {
        'cities': sorted(cities),
        'airports': sorted(airports),
        'countries': sorted(countries),
        'doc_types': sorted(doc_types),
        'topic_keys': sorted(topic_keys)
    }

candidates = fetch_candidates()


In [11]:
# 3) Load question text
def get_question_text(qnum: int) -> str:
    base = ANNOT_DIR / f'q_{qnum}'
    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            q = (data.get('question') or '').strip()
            if q:
                return q
    raise FileNotFoundError(f'Question text not found for q_{qnum}')

QNUM = 26
QUESTION = get_question_text(QNUM)
QUESTION


'In which city in France are the most documents requested?'

In [12]:
# 4) Intent detection (simple rules)
def detect_intent(question: str) -> str:
    q = question.lower()
    if 'question' in q or 'asked' in q or 'questioning' in q:
        return 'questioning'
    if any(x in q for x in ['document', 'documents', 'bring', 'show', 'passport control', 'request', 'requested']):
        return 'documentCheck'
    return 'documentCheck'

INTENT = detect_intent(QUESTION)
INTENT


'documentCheck'

In [13]:
# 5) Scope extraction (explicit matches only)
def normalize_text(s: str) -> str:
    return re.sub(r'[^a-z0-9]+', ' ', s.lower()).strip()

def key_to_surface(key: str) -> str:
    if '_' in key:
        return key.split('_', 1)[1].replace('_', ' ')
    return key

def match_keys(question: str, keys):
    q = normalize_text(question)
    out = []
    for k in keys:
        surface = normalize_text(key_to_surface(k))
        if surface and f' {surface} ' in f' {q} ':
            out.append(k)
    return sorted(set(out))

scope = {
    'cities': match_keys(QUESTION, candidates['cities']),
    'airports': match_keys(QUESTION, candidates['airports']),
    'countries': match_keys(QUESTION, candidates['countries'])
}
scope


{'cities': [], 'airports': [], 'countries': ['country_france']}

In [14]:
# 6) Constraints and aggregation detection
def detect_aggregate(question: str) -> dict:
    q = question.lower()
    if any(x in q for x in ['most', 'least', 'highest', 'lowest', 'top', 'fewest', 'how many', 'number of']):
        if 'country' in q:
            return {'group_by': 'country', 'order': 'desc', 'top_k': 1}
        if 'city' in q:
            return {'group_by': 'city', 'order': 'desc', 'top_k': 1}
    return None

def extract_doc_types(question: str, doc_types):
    q = normalize_text(question)
    out = []
    for k in doc_types:
        surface = normalize_text(k.replace('_', ' '))
        if surface and f' {surface} ' in f' {q} ':
            out.append(k)
    return sorted(set(out))

def extract_topic_keys(question: str, topic_keys):
    return match_keys(question, topic_keys)

aggregate = detect_aggregate(QUESTION)
doc_keys = extract_doc_types(QUESTION, candidates['doc_types'])
topic_keys = extract_topic_keys(QUESTION, candidates['topic_keys'])
aggregate, doc_keys, topic_keys


({'group_by': 'city', 'order': 'desc', 'top_k': 1}, [], [])

In [15]:
# 7) Build query graph object (intent + scope + constraints)
def build_query_graph(question, intent, scope, doc_keys, topic_keys, aggregate):
    return {
        'question': question,
        'intent': intent,
        'scope': scope,
        'doc_keys': doc_keys,
        'topic_keys': topic_keys,
        'aggregate': aggregate,
    }

QUERY_GRAPH = build_query_graph(QUESTION, INTENT, scope, doc_keys, topic_keys, aggregate)
QUERY_GRAPH


{'question': 'In which city in France are the most documents requested?',
 'intent': 'documentCheck',
 'scope': {'cities': [], 'airports': [], 'countries': ['country_france']},
 'doc_keys': [],
 'topic_keys': [],
 'aggregate': {'group_by': 'city', 'order': 'desc', 'top_k': 1}}

In [16]:
# 8) Compile query graph to Cypher
def compile_documentcheck_cypher(qg):
    agg = qg.get('aggregate')
    if agg:
        group_by = agg.get('group_by')
        if group_by == 'country':
            return '''
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (e)-[:atCity]->(c:City)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)-[:locatedInCity]->(:City)-[:locatedInCountry]->(co3:Country)
WITH e, coalesce(co, co2, co3) AS country
WHERE country IS NOT NULL
  AND (size($countries)=0 OR country.key IN $countries)
MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
RETURN country.key AS country, count(*) AS requested_docs
ORDER BY requested_docs DESC
'''.strip()
        if group_by == 'city':
            return '''
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (a)-[:locatedInCity]->(ac:City)
WITH e, c, ac, a
WITH e, coalesce(c, ac) AS city, a
OPTIONAL MATCH (city)-[:locatedInCountry]->(co:Country)
WHERE city IS NOT NULL
  AND (size($cities)=0 OR city.key IN $cities)
  AND (size($airports)=0 OR a.key IN $airports)
  AND (size($countries)=0 OR co.key IN $countries)
MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
RETURN city.key AS city, count(*) AS requested_docs
ORDER BY requested_docs DESC
'''.strip()
        raise ValueError('Unsupported aggregate group_by')

    # non-aggregate: return triples
    return '''
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3
WHERE (size($airports)=0 OR a.key IN $airports)
  AND (size($cities)=0 OR c.key IN $cities OR c2.key IN $cities)
  AND (size($countries)=0 OR co.key IN $countries OR co2.key IN $countries OR co3.key IN $countries)
OPTIONAL MATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)
OPTIONAL MATCH (dc)-[:requestedDocument]->(di:DocumentInstance)
WITH e,a,c,co,c2,co2,co3,dc,di,di.documentType AS di_type
CALL (e,dc,di,di_type,a,c,co,c2,co2,co3) {
  WITH e,dc WHERE e IS NOT NULL AND dc IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasDocumentCheck' AS p, labels(dc)[0] AS o_label, dc.key AS o_key
  UNION
  WITH dc,di WHERE dc IS NOT NULL AND di IS NOT NULL
  RETURN DISTINCT labels(dc)[0] AS s_label, dc.key AS s_key, 'requestedDocument' AS p, labels(di)[0] AS o_label, di.key AS o_key
  UNION
  WITH di,di_type WHERE di IS NOT NULL AND di_type IS NOT NULL
  RETURN DISTINCT labels(di)[0] AS s_label, di.key AS s_key, 'documentType' AS p, 'Literal' AS o_label, di_type AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()

def compile_questioning_cypher(qg):
    # Similar to documentCheck but for questioning
    return '''
MATCH (e:Encounter)
OPTIONAL MATCH (e)-[:atAirport]->(a:Airport)
OPTIONAL MATCH (e)-[:atCity]->(c:City)
OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
OPTIONAL MATCH (a)-[:locatedInCity]->(c2:City)
OPTIONAL MATCH (a)-[:locatedInCountry]->(co2:Country)
OPTIONAL MATCH (c)-[:locatedInCountry]->(co3:Country)
WITH e,a,c,co,c2,co2,co3
WHERE (size($airports)=0 OR a.key IN $airports)
  AND (size($cities)=0 OR c.key IN $cities OR c2.key IN $cities)
  AND (size($countries)=0 OR co.key IN $countries OR co2.key IN $countries OR co3.key IN $countries)
MATCH (e)-[:hasQuestioning]->(q:Questioning)
WITH e,a,c,co,c2,co2,co3,q,q.topicKey AS q_topic
CALL (e,q,q_topic,a,c,co,c2,co2,co3) {
  WITH e,q WHERE e IS NOT NULL AND q IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'hasQuestioning' AS p, labels(q)[0] AS o_label, q.key AS o_key
  UNION
  WITH q,q_topic WHERE q IS NOT NULL AND q_topic IS NOT NULL
  RETURN DISTINCT labels(q)[0] AS s_label, q.key AS s_key, 'topicKey' AS p, 'Literal' AS o_label, q_topic AS o_key
  UNION
  WITH e,c WHERE e IS NOT NULL AND c IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCity' AS p, labels(c)[0] AS o_label, c.key AS o_key
  UNION
  WITH e,co WHERE e IS NOT NULL AND co IS NOT NULL
  RETURN DISTINCT labels(e)[0] AS s_label, e.key AS s_key, 'atCountry' AS p, labels(co)[0] AS o_label, co.key AS o_key
  UNION
  WITH a,c2 WHERE a IS NOT NULL AND c2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCity' AS p, labels(c2)[0] AS o_label, c2.key AS o_key
  UNION
  WITH c,co3 WHERE c IS NOT NULL AND co3 IS NOT NULL
  RETURN DISTINCT labels(c)[0] AS s_label, c.key AS s_key, 'locatedInCountry' AS p, labels(co3)[0] AS o_label, co3.key AS o_key
  UNION
  WITH a,co2 WHERE a IS NOT NULL AND co2 IS NOT NULL
  RETURN DISTINCT labels(a)[0] AS s_label, a.key AS s_key, 'locatedInCountry' AS p, labels(co2)[0] AS o_label, co2.key AS o_key
}
RETURN DISTINCT s_label, s_key, p, o_label, o_key
'''.strip()

def compile_to_cypher(qg):
    if qg['intent'] == 'documentCheck':
        return compile_documentcheck_cypher(qg)
    if qg['intent'] == 'questioning':
        return compile_questioning_cypher(qg)
    raise ValueError('Unsupported intent')

cypher = compile_to_cypher(QUERY_GRAPH)
cypher


'MATCH (e:Encounter)\nOPTIONAL MATCH (e)-[:atCity]->(c:City)\nOPTIONAL MATCH (e)-[:atAirport]->(a:Airport)\nOPTIONAL MATCH (a)-[:locatedInCity]->(ac:City)\nWITH e, c, ac, a\nWITH e, coalesce(c, ac) AS city, a\nOPTIONAL MATCH (city)-[:locatedInCountry]->(co:Country)\nWHERE city IS NOT NULL\n  AND (size($cities)=0 OR city.key IN $cities)\n  AND (size($airports)=0 OR a.key IN $airports)\n  AND (size($countries)=0 OR co.key IN $countries)\nMATCH (e)-[:hasDocumentCheck]->(dc:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)\nRETURN city.key AS city, count(*) AS requested_docs\nORDER BY requested_docs DESC'

In [17]:
# 9) Execute Cypher and retrieve
params = {
    'cities': QUERY_GRAPH['scope']['cities'],
    'airports': QUERY_GRAPH['scope']['airports'],
    'countries': QUERY_GRAPH['scope']['countries'],
}
rows = run_cypher(cypher, params)
rows[:5]


[{'city': 'city_paris', 'requested_docs': 13},
 {'city': 'city_lyon', 'requested_docs': 4},
 {'city': 'city_nice', 'requested_docs': 1},
 {'city': 'city_marseille', 'requested_docs': 1}]

In [18]:
# 10) Postprocess / evaluate (optional)
def triples_to_set(rows):
    return {
        (r.get('s_label'), r.get('s_key'), r.get('p'), r.get('o_label'), r.get('o_key'))
        for r in rows
        if all(k in r for k in ['s_label','s_key','p','o_label','o_key'])
    }

def load_gold(qnum: int):
    triples = []
    base = ANNOT_DIR / f'q_{qnum}'
    for folder in ['fr_1', 'fr_2', 'germany', 'italy', 'spain']:
        p = base / folder
        if not p.exists():
            continue
        for f in sorted(p.glob('annotation_*.json')):
            data = json.loads(f.read_text(encoding='utf-8'))
            for t in data.get('triples', []):
                s = t.get('s', {})
                o = t.get('o', {})
                triples.append((s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key')))
    return set(triples)

if rows and all(k in rows[0] for k in ['s_label','s_key','p','o_label','o_key']):
    retrieved = triples_to_set(rows)
    gold = load_gold(QNUM)
    tp = len(retrieved & gold)
    precision = tp / len(retrieved) if retrieved else 0.0
    recall = tp / len(gold) if gold else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    print('retrieved', len(retrieved))
    print('gold', len(gold))
    print('precision', precision)
    print('recall', recall)
    print('f1', f1)
else:
    print('Aggregate output:', rows[:5])


Aggregate output: [{'city': 'city_paris', 'requested_docs': 13}, {'city': 'city_lyon', 'requested_docs': 4}, {'city': 'city_nice', 'requested_docs': 1}, {'city': 'city_marseille', 'requested_docs': 1}]
